# 02 Data Cleaning

## Objective

Clean the raw Telco churn dataset so key fields are ready for downstream analysis and modeling. This notebook focuses on the known `TotalCharges` type issue, validates the cleaned result, and saves a cleaned dataset for later stages.


## Imports


In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import load_raw_data
from src.data.data_validation import validate_raw_data


## Load And Validate Raw Data


In [ ]:
df = load_raw_data()
validate_raw_data(df)
df_clean = df.copy()

print("Raw dataset loaded and validated successfully.")
print(f"Data source: {project_root / 'data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'}")
print(f"Shape: {df_clean.shape}")


## Inspect Cleaning Targets


In [ ]:
blank_total_charges = df_clean['TotalCharges'].astype(str).str.strip().eq('')

print(f"TotalCharges dtype before cleaning: {df_clean['TotalCharges'].dtype}")
print(f"Blank TotalCharges rows: {blank_total_charges.sum()}")

df_clean.loc[blank_total_charges, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head(11)


## Clean TotalCharges

The blank `TotalCharges` rows belong to customers with `tenure == 0`, so replacing blanks with `0` is consistent with a newly started account that has not accumulated historical charges yet.


In [ ]:
df_clean['TotalCharges'] = df_clean['TotalCharges'].replace(' ', pd.NA)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0.0)

print(f"TotalCharges dtype after cleaning: {df_clean['TotalCharges'].dtype}")
print(f"Missing TotalCharges after cleaning: {df_clean['TotalCharges'].isna().sum()}")
print(f"Rows with tenure == 0 and TotalCharges == 0: {((df_clean['tenure'] == 0) & (df_clean['TotalCharges'] == 0)).sum()}")


## Verify Cleaned Dataset


In [ ]:
df_clean.info()

missing_values = df_clean.isnull().sum()
missing_values = missing_values[missing_values > 0]

if missing_values.empty:
    print("No missing values found in the cleaned dataset.")
else:
    display(missing_values.sort_values(ascending=False))


## Save Cleaned Data


In [ ]:
output_path = project_root / 'data/interim/telco_churn_cleaned.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"Saved shape: {df_clean.shape}")
